# Part 26 — Vectorless RAG: PageIndex

Traditional RAG pipelines embed documents into vectors and retrieve by cosine similarity. PageIndex replaces similarity lookup with **LLM-driven reasoning over a hierarchical tree index** — no vector database required.

```
TWO-PHASE PIPELINE
══════════════════════════════════════════════════════════════════════

PHASE 1: INDEX GENERATION (one-time, offline)
  PDF / Markdown
       │
       ▼
  ┌─────────────┐   detect TOC    ┌──────────────┐   LLM summaries  ┌──────────────────┐
  │  Parse Text │ ──────────────► │  Build Tree  │ ────────────────► │  Enrich Nodes    │
  │  + Tokens   │                 │  (hierarchy) │                   │  (node_id, range)│
  └─────────────┘                 └──────────────┘                   └──────────────────┘
                                                                               │
                                                                    JSON tree index saved

PHASE 2: REASONING-BASED RETRIEVAL (per-query, online)
  User Query
       │
       ▼
  ┌──────────────┐   select nodes  ┌─────────────────┐   extract text  ┌─────────────┐
  │  LLM reads   │ ──────────────► │  Fetch content  │ ───────────────► │  Generate   │
  │  tree TOC    │                 │  from page range│                   │  Answer     │
  └──────────────┘                 └─────────────────┘                   └─────────────┘
       ▲                                                                       │
       └─────────── loop if answer insufficient ◄────────────────────────────┘
```

| Section | Content |
|---|---|
| 1 | Problems with Vector RAG |
| 2 | Architecture comparison visualization |
| 3 | Tree index data structure |
| 4 | Tree visualization |
| 5 | Index generation — parsing + tree building |
| 6 | Index generation — node enrichment |
| 7 | Full mock index builder |
| 8 | Retrieval — LLM-driven tree search |
| 9 | Retrieval path visualization |
| 10 | Real PageIndex integration |
| 11 | Performance comparison visualization |
| 12 | When to use / not use |
| 13 | Summary |

## 1 — Problems with Vector RAG

Vector RAG retrieves chunks by cosine similarity between query and chunk embeddings. Similarity is a proxy for relevance, and it fails in predictable ways:

| Failure Mode | Root Cause | Example |
|---|---|---|
| **Query-document mismatch** | Query uses different vocabulary than document | "profitability" vs "net income margin" |
| **Chunking artifacts** | Fixed-size splits cut across logical boundaries | Answer spans pages 14–15; chunk ends at page 14 line 40 |
| **Cross-reference blindness** | Chunks are independent; "See Appendix G" is not followed | Table on page 5 references footnote definition on page 47 |
| **Similarity ≠ relevance** | High cosine score doesn't guarantee the chunk answers the query | Document contains "revenue" 200 times; wrong chunks returned |
| **Flat structure loss** | Hierarchy (chapter → section → subsection) destroyed at chunk time | "What does Section 3.2 say about X?" — section boundaries gone |
| **No multi-hop reasoning** | Single retrieval pass; cannot refine based on partial information | Question requires combining information from 3 different sections |

**PageIndex addresses all six by replacing embedding lookup with LLM reasoning over a preserved document hierarchy.**

<div align="center" style="display: flex; justify-content: center; gap: 20px;">
    <img src="https://raw.githubusercontent.com/sprashant433/GenAI/main/images/vectorless_rag_comparison.png" style="width:50%; height:auto;" />
</div>

## 3 — Tree Index Data Structure

The tree index is a recursive JSON structure. Each node maps to a document section with a page/line range. No vectors are stored — only human-readable summaries the LLM uses for navigation.

```json
{
  "title": "Annual Report 2024",
  "node_id": "0",
  "summary": "Full annual report covering financials, operations, and risk factors for FY2024.",
  "start_index": 1,
  "end_index": 148,
  "nodes": [
    {
      "title": "1. Business Overview",
      "node_id": "1",
      "summary": "Company description, segment breakdown, and key products sold in FY2024.",
      "start_index": 1,
      "end_index": 18,
      "nodes": [
        {
          "title": "1.1 Segments",
          "node_id": "1.1",
          "summary": "Three business segments: Cloud (52%), Hardware (31%), Services (17%).",
          "start_index": 5,
          "end_index": 12,
          "nodes": []
        }
      ]
    },
    {
      "title": "2. Financial Statements",
      "node_id": "2",
      "summary": "Consolidated income statement, balance sheet, and cash flow for FY2024.",
      "start_index": 19,
      "end_index": 74,
      "nodes": [...]
    }
  ]
}
```

| Field | Type | Purpose |
|---|---|---|
| `node_id` | string | Hierarchical dot-notation (`"1.2.3"`) — used by LLM to refer to nodes |
| `title` | string | Section heading from the original document |
| `summary` | string | 1-sentence LLM-generated description — the **retrieval signal** |
| `start_index` | int | First page (PDF) or line number (Markdown) |
| `end_index` | int | Last page/line — defines content extraction range |
| `nodes` | list | Recursive child nodes; empty list at leaf level |

**Key design decisions:**
- Splits occur at **natural semantic boundaries** (headings), not token count
- Leaf nodes are bounded by `max_pages_per_node` (default 10) and `max_tokens_per_node` (default 20,000)
- Oversized leaf nodes are **recursively re-indexed** as sub-documents
- Summaries are generated **concurrently** via `asyncio.gather()` to keep indexing fast

<div align="center" style="display: flex; justify-content: center; gap: 20px;">
    <img src="https://raw.githubusercontent.com/sprashant433/GenAI/main/images/vectorless_rag_tree.png" style="width:50%; height:auto;" />
</div>

## 5 — Index Generation: Parsing + Tree Building

**Stage 1 — Parsing:** Extract text and count tokens per page/section.

**Stage 2 — Tree building:** Three modes depending on document structure:

| Mode | Trigger | Algorithm |
|---|---|---|
| `TOC_WITH_PAGES` | TOC detected + page numbers present | Map TOC titles → pages; detect printed/physical page offset |
| `TOC_NO_PAGES` | TOC detected, no page numbers | LLM call per section: "On which page does *Introduction* begin?" |
| `NO_TOC` | No TOC found | Progressive scanning — LLM segments document into logical sections |

**Fallback logic:** If accuracy < 60% (verified against sampled pages), cascade down to next mode. `meta_processor()` orchestrates all three modes with this fallback chain.

**Stage 3 — Enrichment:** After tree is built, add `node_id` (dot-notation), extract page text per node, generate one-sentence summaries concurrently with `asyncio.gather()`.

In [ ]:
import re
from dataclasses import dataclass, field
from typing import List, Optional

# ── Data model ────────────────────────────────────────────────────────────────
@dataclass
class TOCNode:
    title: str
    start_index: int
    end_index: int
    node_id: str = ""
    summary: str = ""
    nodes: List['TOCNode'] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "title": self.title,
            "node_id": self.node_id,
            "summary": self.summary,
            "start_index": self.start_index,
            "end_index": self.end_index,
            "nodes": [c.to_dict() for c in self.nodes]
        }

# ── Stage 1: Parse document into pages ───────────────────────────────────────
def parse_pages(text: str) -> List[str]:
    """Split text into pages by form-feed or double newline blocks."""
    pages = re.split(r'\f|\n{3,}', text.strip())
    return [p.strip() for p in pages if p.strip()]

def get_page_tokens(pages: List[str]) -> List[int]:
    """Approximate token count per page (≈ words * 1.3)."""
    return [int(len(p.split()) * 1.3) for p in pages]

# ── Stage 2a: Build tree when TOC + page numbers are present ─────────────────
def parse_toc_line(line: str) -> Optional[tuple]:
    """Extract (indent_level, title, page_number) from a TOC line.
    Handles formats like: '  1.2 Methods ............. 24'
    """
    m = re.match(r'^(\s*)(\d[\d.]*)\s+(.*?)\s*\.{2,}\s*(\d+)\s*$', line)
    if m:
        indent = len(m.group(1)) // 2
        section_num = m.group(2)
        title = m.group(3).strip()
        page = int(m.group(4))
        return (indent, f"{section_num} {title}", page)
    return None

def build_tree_from_toc(toc_lines: List[str], total_pages: int) -> List[TOCNode]:
    """Build a list of top-level TOCNodes from parsed TOC lines."""
    entries = [parse_toc_line(l) for l in toc_lines if parse_toc_line(l)]
    if not entries:
        return []

    nodes: List[TOCNode] = []
    stack: List[TOCNode] = []

    for i, (depth, title, start_page) in enumerate(entries):
        end_page = entries[i + 1][2] - 1 if i + 1 < len(entries) else total_pages
        node = TOCNode(title=title, start_index=start_page, end_index=end_page)

        # Pop stack until we find the right parent depth
        while len(stack) > depth:
            stack.pop()

        if stack:
            stack[-1].nodes.append(node)
        else:
            nodes.append(node)

        stack.append(node)

    return nodes

# ── Stage 3: Assign node_ids ──────────────────────────────────────────────────
def assign_node_ids(nodes: List[TOCNode], prefix: str = "") -> None:
    for i, node in enumerate(nodes, start=1):
        node.node_id = f"{prefix}{i}" if prefix else str(i)
        if node.nodes:
            assign_node_ids(node.nodes, prefix=f"{node.node_id}.")

# ── Demo: Parse a synthetic TOC ──────────────────────────────────────────────
SAMPLE_TOC = """
  1 Business Overview ............. 1
    1.1 Segments .................. 5
    1.2 Products .................. 13
  2 Financial Statements .......... 19
    2.1 Income Statement .......... 19
    2.2 Balance Sheet ............. 36
    2.3 Cash Flow ................. 56
  3 Risk Factors .................. 75
    3.1 Regulatory Risk ........... 75
    3.2 Market Risk ............... 90
  4 Appendix ...................... 111
""".strip().splitlines()

nodes = build_tree_from_toc(SAMPLE_TOC, total_pages=148)
assign_node_ids(nodes)

print("Parsed TOC → TOCNode tree:")
def print_tree(nodes, indent=0):
    for n in nodes:
        print(f"{'  ' * indent}[{n.node_id}] {n.title}  (p{n.start_index}–{n.end_index})")
        print_tree(n.nodes, indent + 1)

print_tree(nodes)

## 6 — Index Generation: Node Enrichment

After tree structure is built, each node gets:
1. **Text content** — extracted from the page range via `get_page_content(start, end)`
2. **LLM summary** — one sentence describing what the section contains, used as the retrieval signal
3. **Token count** — determines if the node needs recursive splitting

Summaries are generated **concurrently** using `asyncio.gather()` to avoid sequential LLM call latency. For a 50-node document, this reduces enrichment time from ~50s to ~5s (10x speedup with 10 concurrent requests).

In [ ]:
import asyncio
import json
import time

# ── Simulated document pages ───────────────────────────────────────────────
FAKE_PAGES = {
    "1":   "Business Overview: The company operates in three core segments...",
    "1.1": "Segments: Cloud revenue $4.4B (52%), Hardware $2.6B (31%), Services $1.4B (17%)...",
    "1.2": "Products: Introduced CloudOS v3 and EdgeBox Pro in Q2 2024...",
    "2":   "Financial Statements: Consolidated results for fiscal year ended Dec 31 2024...",
    "2.1": "Income Statement: Total revenue $8.4B, up 12% YoY. Net income $1.2B. EPS $3.41...",
    "2.2": "Balance Sheet: Total assets $42.1B. Goodwill $8.2B. Total liabilities $28.3B...",
    "2.3": "Cash Flow: Operating cash flow $2.1B. CapEx $0.9B. Free cash flow $1.2B...",
    "3":   "Risk Factors: Four material risk categories identified by management...",
    "3.1": "Regulatory Risk: GDPR compliance costs +18% YoY; two SEC inquiries pending...",
    "3.2": "Market Risk: 38% revenue outside USD; 100bps rate increase → $40M interest impact...",
    "4":   "Appendix: Non-GAAP reconciliation, auditor independence disclosure, glossary...",
}

# ── Simulate LLM summary generation (mocked) ─────────────────────────────────
MOCK_SUMMARIES = {
    "1":   "Three-segment business (Cloud, Hardware, Services) with Cloud as primary revenue driver at 52%.",
    "1.1": "Revenue breakdown: Cloud 52% ($4.4B), Hardware 31% ($2.6B), Services 17% ($1.4B).",
    "1.2": "New products CloudOS v3 and EdgeBox Pro launched in Q2 2024 with premium pricing tiers.",
    "2":   "Full consolidated financials for FY2024 including income statement, balance sheet, and cash flow.",
    "2.1": "Revenue $8.4B (+12% YoY), net income $1.2B, diluted EPS $3.41.",
    "2.2": "Total assets $42.1B, total liabilities $28.3B, shareholders' equity $13.8B.",
    "2.3": "Operating cash flow $2.1B, capex $0.9B, free cash flow $1.2B for FY2024.",
    "3":   "Four material risk categories: regulatory, competitive, macroeconomic, cybersecurity.",
    "3.1": "GDPR and SEC compliance obligations with two pending inquiries as of report date.",
    "3.2": "FX exposure on 38% non-USD revenue; interest rate sensitivity disclosed at $40M per 100bps.",
    "4":   "Non-GAAP reconciliation tables, auditor notes, and financial glossary.",
}

async def generate_summary(node_id: str, page_text: str) -> str:
    """Simulate async LLM call with slight delay."""
    await asyncio.sleep(0.05)
    return MOCK_SUMMARIES.get(node_id, f"Section {node_id} content.")

async def enrich_nodes(nodes: List[TOCNode]) -> None:
    """Concurrently generate summaries for all leaf nodes."""
    async def enrich_one(node: TOCNode):
        page_text = FAKE_PAGES.get(node.node_id, f"Content for node {node.node_id}.")
        node.summary = await generate_summary(node.node_id, page_text)
        # Recurse into children
        if node.nodes:
            await asyncio.gather(*[enrich_one(c) for c in node.nodes])

    await asyncio.gather(*[enrich_one(n) for n in nodes])

# Run enrichment
start = time.time()
asyncio.run(enrich_nodes(nodes))
elapsed = time.time() - start

print(f"Enrichment completed in {elapsed:.2f}s (concurrent)\n")
print("Enriched tree (showing summaries):")
def print_enriched(nodes, indent=0):
    for n in nodes:
        print(f"{'  ' * indent}[{n.node_id}] {n.title}")
        print(f"{'  ' * indent}  → {n.summary}")
        print_enriched(n.nodes, indent + 1)

print_enriched(nodes)

# Serialize to JSON
index_json = {
    "title": "Annual Report 2024",
    "node_id": "0",
    "summary": "Full annual report covering financials, operations, and risk factors.",
    "start_index": 1,
    "end_index": 148,
    "nodes": [n.to_dict() for n in nodes]
}
print(f"\nJSON index size: {len(json.dumps(index_json))} chars")
print(f"Token estimate : ~{len(json.dumps(index_json).split()) * 130 // 100} tokens (vs ~{148 * 400} tokens for full document)")

## 8 — Retrieval: LLM-Driven Tree Search

The retrieval loop has **five steps** per iteration:

```
1. BUILD TOC VIEW  — flatten tree into a list of (node_id, summary) pairs
2. LLM SELECTS     — model reasons: "to answer this query, I need nodes [X, Y]"
3. FETCH CONTENT   — extract page text for the selected node_id ranges
4. ASSESS          — LLM checks: "does this content answer the query?"
5. ANSWER / LOOP   — if sufficient → generate answer; else → repeat with new nodes
```

**Key properties:**
- The LLM never sees the full document — only the tree structure (summaries) + selected page content
- Each iteration sends ~500–2,000 tokens (tree view) + ~3,000–8,000 tokens (page content)
- For most queries, 1–2 iterations suffice; complex multi-hop queries may need 3–4
- Node selection is **traceable**: the answer cites which `node_id` values were used

In [ ]:
from typing import Dict, Tuple
import textwrap

# ── Flatten tree into TOC view (what the LLM sees) ───────────────────────────
def build_toc_view(index: dict, depth: int = 0) -> str:
    indent = "  " * depth
    line = f"{indent}[{index['node_id']}] {index['title']}  — {index['summary']}"
    lines = [line]
    for child in index.get('nodes', []):
        lines.append(build_toc_view(child, depth + 1))
    return "\n".join(lines)

# ── Get node content by node_id ───────────────────────────────────────────────
def get_node_by_id(index: dict, node_id: str) -> dict:
    if index['node_id'] == node_id:
        return index
    for child in index.get('nodes', []):
        result = get_node_by_id(child, node_id)
        if result:
            return result
    return None

def fetch_content(index: dict, node_ids: list) -> Dict[str, str]:
    """Return page text for each requested node_id."""
    content = {}
    for nid in node_ids:
        node = get_node_by_id(index, nid)
        if node:
            content[nid] = FAKE_PAGES.get(nid, f"[content for node {nid}, pages {node['start_index']}–{node['end_index']}]")
    return content

# ── Simulated LLM reasoning (mocked) ─────────────────────────────────────────
def llm_select_nodes(query: str, toc_view: str) -> Tuple[list, str]:
    """Mock: LLM reads TOC and returns node_ids to fetch."""
    query_lower = query.lower()
    if "revenue" in query_lower or "income" in query_lower or "eps" in query_lower:
        return ["2.1"], "Query is about financial results → Income Statement (2.1)"
    elif "cash" in query_lower or "capex" in query_lower:
        return ["2.3"], "Query is about cash position → Cash Flow (2.3)"
    elif "risk" in query_lower or "regulatory" in query_lower:
        return ["3.1", "3.2"], "Query is about risk → Risk Factors (3.1, 3.2)"
    elif "segment" in query_lower or "cloud" in query_lower:
        return ["1.1"], "Query is about segments → Business Overview / Segments (1.1)"
    elif "balance" in query_lower or "assets" in query_lower or "liabilities" in query_lower:
        return ["2.2"], "Query is about balance sheet → Balance Sheet (2.2)"
    else:
        return ["1", "2", "3"], "Broad query → fetch top-level chapters"

def llm_assess_sufficiency(query: str, content: Dict[str, str]) -> Tuple[bool, str]:
    """Mock: LLM checks if fetched content answers the query."""
    combined = " ".join(content.values()).lower()
    keywords = [w for w in query.lower().split() if len(w) > 4]
    hits = sum(1 for kw in keywords if kw in combined)
    sufficient = hits >= max(1, len(keywords) // 2)
    return sufficient, f"{hits}/{len(keywords)} query keywords found in fetched content"

def llm_generate_answer(query: str, content: Dict[str, str]) -> str:
    """Mock: LLM generates answer from fetched content."""
    node_refs = ", ".join(f"[{nid}]" for nid in content)
    excerpt = list(content.values())[0][:120] + "..."
    return f"Based on {node_refs}: {excerpt}"

# ── Retrieval loop ────────────────────────────────────────────────────────────
def retrieve_and_answer(query: str, index: dict, max_iterations: int = 3) -> dict:
    toc_view = build_toc_view(index)
    visited_nodes = set()
    history = []

    for iteration in range(1, max_iterations + 1):
        node_ids, reasoning = llm_select_nodes(query, toc_view)
        new_ids = [n for n in node_ids if n not in visited_nodes]
        visited_nodes.update(new_ids)

        content = fetch_content(index, new_ids)
        sufficient, assessment = llm_assess_sufficiency(query, content)

        step = {
            "iteration": iteration,
            "reasoning": reasoning,
            "fetched_nodes": new_ids,
            "assessment": assessment,
            "sufficient": sufficient,
        }
        history.append(step)

        if sufficient:
            answer = llm_generate_answer(query, content)
            return {"query": query, "answer": answer, "iterations": iteration,
                    "nodes_used": list(visited_nodes), "history": history}

    # Fallback: answer from all collected content
    all_content = fetch_content(index, list(visited_nodes))
    return {"query": query, "answer": llm_generate_answer(query, all_content),
            "iterations": max_iterations, "nodes_used": list(visited_nodes), "history": history}

# ── Run three example queries ─────────────────────────────────────────────────
QUERIES = [
    "What was the company's revenue and EPS in FY2024?",
    "What regulatory risks does the company face?",
    "How does the Cloud segment compare to Hardware in revenue?",
]

for q in QUERIES:
    result = retrieve_and_answer(q, index_json)
    print(f"Query : {q}")
    print(f"Nodes : {result['nodes_used']}  ({result['iterations']} iteration(s))")
    print(f"Answer: {result['answer'][:120]}...")
    print()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

# ── Visualize retrieval path for the revenue query ───────────────────────────
query = "What was the company's revenue and EPS in FY2024?"
result = retrieve_and_answer(query, index_json)

fig, ax = plt.subplots(figsize=(15, 9))
fig.patch.set_facecolor('#0f0f0f')
ax.set_facecolor('#1a1a2e')
ax.axis('off')
ax.set_xlim(0, 15)
ax.set_ylim(0, 9)

# ── Node layout: (node_id, x, y, label, depth) ───────────────────────────────
LAYOUT = {
    "0":   (7.5, 8.2, "Annual Report 2024\n[0]", 0),
    "1":   (2.5, 6.5, "1. Business\nOverview [1]", 1),
    "2":   (7.5, 6.5, "2. Financial\nStatements [2]", 1),
    "3":   (11.5, 6.5, "3. Risk\nFactors [3]", 1),
    "4":   (14.0, 6.5, "4. Appendix\n[4]", 1),
    "1.1": (1.5, 4.5, "1.1 Segments\n[1.1]", 2),
    "1.2": (3.5, 4.5, "1.2 Products\n[1.2]", 2),
    "2.1": (6.0, 4.5, "2.1 Income\nStatement [2.1]", 2),
    "2.2": (8.0, 4.5, "2.2 Balance\nSheet [2.2]", 2),
    "2.3": (10.0, 4.5, "2.3 Cash\nFlow [2.3]", 2),
    "3.1": (10.8, 4.5, "3.1 Regulatory\n[3.1]", 2),
    "3.2": (12.5, 4.5, "3.2 Market\nRisk [3.2]", 2),
}

EDGES = [
    ("0","1"),("0","2"),("0","3"),("0","4"),
    ("1","1.1"),("1","1.2"),
    ("2","2.1"),("2","2.2"),("2","2.3"),
    ("3","3.1"),("3","3.2"),
]

DEPTH_COLORS = {0: ('#1e3a5f','#4a9eff'), 1: ('#2d4a1e','#90c47a'), 2: ('#3a2a10','#f0a030')}
VISITED = set(result['nodes_used'])

def node_color(nid):
    if nid in VISITED:
        return ('#1a3a1a', '#40e040')  # highlighted green
    depth = LAYOUT[nid][3]
    return DEPTH_COLORS[depth]

# Draw edges
for (src, dst) in EDGES:
    x0, y0 = LAYOUT[src][0], LAYOUT[src][1]
    x1, y1 = LAYOUT[dst][0], LAYOUT[dst][1]
    lw = 2.5 if (src in VISITED or dst in VISITED) else 0.8
    color = '#40e040' if (src in VISITED and dst in VISITED) else '#333'
    ax.plot([x0, x1], [y0 - 0.4, y1 + 0.4], color=color, lw=lw, zorder=1)

# Draw nodes
for nid, (x, y, label, depth) in LAYOUT.items():
    fc, ec = node_color(nid)
    lw = 2.5 if nid in VISITED else 1.0
    w, h = 2.2 - depth * 0.1, 0.72
    box = mpatches.FancyBboxPatch((x - w/2, y - h/2), w, h,
        boxstyle="round,pad=0.08", linewidth=lw, edgecolor=ec, facecolor=fc, zorder=2)
    ax.add_patch(box)
    tc = '#40e040' if nid in VISITED else '#aaa'
    ax.text(x, y, label, color=tc, fontsize=7.5, ha='center', va='center',
        fontweight='bold' if nid in VISITED else 'normal', zorder=3)

# ── Retrieval path annotation ─────────────────────────────────────────────────
ax.text(7.5, 2.8, f'Query: "{query}"',
    color='#60c0c0', fontsize=10, ha='center', va='center', style='italic',
    bbox=dict(boxstyle='round', facecolor='#102a2a', edgecolor='#60c0c0', linewidth=1.2))

ax.text(7.5, 2.1, f"Step 1 — LLM reads TOC summaries (no page content sent yet)",
    color='#f0a030', fontsize=9, ha='center', va='center')
ax.text(7.5, 1.65, f"Step 2 — LLM reasoning: \"query is about revenue/EPS → fetch node [2.1]\"",
    color='#f0c060', fontsize=9, ha='center', va='center')
ax.text(7.5, 1.2, f"Step 3 — Fetch pages 19–35 (Income Statement) → sufficiency: PASS",
    color='#90e090', fontsize=9, ha='center', va='center')
ax.text(7.5, 0.75, f"Result: answered in {result['iterations']} iteration | nodes used: {result['nodes_used']} | ~{35-19+1} pages read of 148 total ({(35-19+1)/148*100:.0f}%)",
    color='#70c070', fontsize=9, ha='center', va='center',
    bbox=dict(boxstyle='round', facecolor='#0a2a0a', edgecolor='#40a040', linewidth=1))

# Legend
visited_patch = mpatches.Patch(facecolor='#1a3a1a', edgecolor='#40e040', label='Visited node')
skipped_patch = mpatches.Patch(facecolor='#2a2a2a', edgecolor='#444', label='Not visited (skipped)')
ax.legend(handles=[visited_patch, skipped_patch], loc='upper left',
    facecolor='#1a1a2e', edgecolor='#555', labelcolor='white', fontsize=9, framealpha=0.7)

ax.set_title(f"Retrieval Path: LLM navigates directly to relevant node — skips 10 of 11 nodes",
    color='white', fontsize=12, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig("vectorless_rag_retrieval_path.png", dpi=150, bbox_inches='tight',
    facecolor=fig.get_facecolor())
plt.show()
print("Saved → images/vectorless_rag_retrieval_path.png")

## 10 — Real PageIndex Integration

Install and use the actual PageIndex library. Requires an OpenAI API key — indexing calls GPT-4o to generate node summaries.

```bash
pip install pageindex openai pypdf2
```

**CLI usage (generate index from a PDF):**
```bash
python3 run_pageindex.py --pdf_path ./annual_report.pdf --model gpt-4o-2024-11-20
# Output: ./results/annual_report_structure.json
```

**Key CLI parameters:**

| Parameter | Default | Effect |
|---|---|---|
| `--max-pages-per-node` | 10 | Max pages per leaf node; larger → fewer nodes, less LLM calls at retrieval |
| `--max-tokens-per-node` | 20000 | Max tokens per node; triggers recursive splitting if exceeded |
| `--toc-check-pages` | 20 | How many pages to scan looking for a TOC |
| `--model` | gpt-4o-2024-11-20 | LLM for TOC detection + summary generation |

In [ ]:
import os
import json
from openai import OpenAI

# ── Setup ─────────────────────────────────────────────────────────────────────
# Requires: pip install pageindex openai pypdf2
# Requires: OPENAI_API_KEY environment variable

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
PDF_PATH = "your_document.pdf"         # replace with actual PDF path
INDEX_PATH = "results/doc_structure.json"

# ── OPTION A: Generate index via CLI (run in terminal) ────────────────────────
# python3 run_pageindex.py --pdf_path your_document.pdf --model gpt-4o-2024-11-20

# ── OPTION B: Load pre-built index and query with OpenAI ─────────────────────
def load_index(path: str) -> dict:
    with open(path) as f:
        return json.load(f)

def build_toc_for_llm(index: dict, depth: int = 0) -> str:
    indent = "  " * depth
    line = f"{indent}[{index['node_id']}] {index['title']} — {index.get('summary','')}"
    parts = [line]
    for child in index.get("nodes", []):
        parts.append(build_toc_for_llm(child, depth + 1))
    return "\n".join(parts)

def get_page_text(index: dict, node_id: str, pages: dict) -> str:
    """pages: dict mapping node_id → extracted text (from PyPDF2)."""
    node = get_node_by_id(index, node_id)
    if not node:
        return ""
    return pages.get(node_id, f"[pages {node['start_index']}–{node['end_index']}]")

def pageindex_rag(query: str, index: dict, pages: dict, client: OpenAI) -> str:
    toc = build_toc_for_llm(index)

    # Step 1: LLM selects nodes
    select_response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": (
                "You are a document navigator. Given a query and a document table of contents "
                "(each entry has a node_id and one-sentence summary), return a JSON list of "
                "node_ids that are most likely to contain the answer. Return ONLY JSON: {\"node_ids\": [...]}"
            )},
            {"role": "user", "content": f"Query: {query}\n\nTable of Contents:\n{toc}"}
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )
    node_ids = json.loads(select_response.choices[0].message.content).get("node_ids", [])

    # Step 2: Fetch content for selected nodes
    context_parts = []
    for nid in node_ids:
        text = get_page_text(index, nid, pages)
        context_parts.append(f"[Node {nid}]\n{text}")
    context = "\n\n".join(context_parts)

    # Step 3: Generate answer
    answer_response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": (
                "Answer the question using only the provided document sections. "
                "Cite the node_id(s) you used in your answer."
            )},
            {"role": "user", "content": f"Question: {query}\n\nDocument sections:\n{context}"}
        ],
        temperature=0,
    )
    return answer_response.choices[0].message.content

# ── Example (requires real index + API key) ────────────────────────────────────
if OPENAI_API_KEY and os.path.exists(INDEX_PATH):
    client = OpenAI(api_key=OPENAI_API_KEY)
    index = load_index(INDEX_PATH)
    pages = {}  # populate with {node_id: "page text"} from PyPDF2

    answer = pageindex_rag(
        query="What is the company's free cash flow for FY2024?",
        index=index,
        pages=pages,
        client=client,
    )
    print(answer)
else:
    print("Skipping live execution — no API key or index file found.")
    print()
    print("To run:")
    print("  1. pip install pageindex openai pypdf2")
    print("  2. export OPENAI_API_KEY=sk-...")
    print("  3. python3 run_pageindex.py --pdf_path your_doc.pdf")
    print("  4. Set INDEX_PATH = 'results/your_doc_structure.json' and re-run this cell")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Performance data: FinanceBench + Token efficiency ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor('#0f0f0f')
for ax in axes:
    ax.set_facecolor('#1a1a2e')

# ── Left: Accuracy comparison on FinanceBench ─────────────────────────────────
ax = axes[0]
methods = ['Naive\nVector RAG', 'RAG +\nReranking', 'Full-doc\nPrompting', 'PageIndex\n(Vectorless)']
accuracy = [49.7, 62.3, 78.1, 98.7]
colors = ['#c03030', '#c07030', '#c0c030', '#30c060']

bars = ax.bar(methods, accuracy, color=colors, edgecolor='#333', linewidth=1.2, width=0.55)

for bar, acc in zip(bars, accuracy):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.2,
        f'{acc}%', color='white', fontsize=11, ha='center', va='bottom', fontweight='bold')

ax.set_ylim(0, 110)
ax.set_ylabel('Accuracy (%)', color='#aaa', fontsize=11)
ax.set_title('FinanceBench QA Accuracy', color='white', fontsize=13, fontweight='bold', pad=10)
ax.tick_params(colors='#aaa', labelsize=9)
ax.spines['bottom'].set_color('#333')
ax.spines['left'].set_color('#333')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.label.set_color('#aaa')
ax.axhline(y=90, color='#666', linestyle='--', linewidth=0.8, alpha=0.5)
ax.text(3.35, 91.5, '90% threshold', color='#666', fontsize=8)

# ── Right: Token efficiency ────────────────────────────────────────────────────
ax = axes[1]
doc_sizes = [50, 100, 200, 500]
full_doc_tokens = [d * 400 for d in doc_sizes]        # ~400 tokens/page
vector_rag_tokens = [d * 400 * 0.25 for d in doc_sizes]  # top-k ~25% of doc
pageindex_tokens = [d * 400 * 0.075 for d in doc_sizes]  # ~7.5% (tree + 1-2 nodes)

x = np.arange(len(doc_sizes))
width = 0.25

b1 = ax.bar(x - width, full_doc_tokens, width, label='Full-doc prompting', color='#c03030', edgecolor='#333')
b2 = ax.bar(x, vector_rag_tokens, width, label='Vector RAG (top-k chunks)', color='#c07030', edgecolor='#333')
b3 = ax.bar(x + width, pageindex_tokens, width, label='PageIndex (tree + 1-2 nodes)', color='#30c060', edgecolor='#333')

ax.set_xticks(x)
ax.set_xticklabels([f'{d}p\ndoc' for d in doc_sizes], color='#aaa', fontsize=9)
ax.set_ylabel('Tokens sent to LLM per query', color='#aaa', fontsize=11)
ax.set_title('Token Efficiency per Query', color='white', fontsize=13, fontweight='bold', pad=10)
ax.legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white', fontsize=8)
ax.tick_params(colors='#aaa')
ax.spines['bottom'].set_color('#333')
ax.spines['left'].set_color('#333')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.label.set_color('#aaa')

# Annotate PageIndex reduction
for i, (fd, pi) in enumerate(zip(full_doc_tokens, pageindex_tokens)):
    reduction = (1 - pi / fd) * 100
    ax.text(x[i] + width, pi + fd * 0.01, f'-{reduction:.0f}%',
        color='#30c060', fontsize=7.5, ha='center', va='bottom', fontweight='bold')

plt.suptitle("PageIndex Performance: 98.7% accuracy on FinanceBench, 92%+ token reduction",
    color='white', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("images/vectorless_rag_performance.png", dpi=150, bbox_inches='tight',
    facecolor=fig.get_facecolor())
plt.show()
print("Saved → images/vectorless_rag_performance.png")

## 12 — When to Use / Not Use PageIndex

| Criterion | Use PageIndex | Use Vector RAG |
|---|---|---|
| **Document structure** | Well-structured (TOC, headings, sections) | Unstructured text, conversational content |
| **Document length** | Long (50+ pages); hard to fit in context | Short docs that fit in a single prompt |
| **Query type** | Specific, section-targeted questions | Broad semantic similarity lookups |
| **Cross-references** | Document uses "see Section X", footnotes, appendices | Independent paragraphs with no cross-refs |
| **Explainability requirement** | High — must cite exact section | Low — approximate answer acceptable |
| **Indexing cost** | Amortized over many queries (re-used index) | Per-chunk embedding is cheap and one-time |
| **Latency requirement** | Tolerates 2–5 LLM calls per query (~3–10s) | Sub-second retrieval via vector lookup |
| **Infrastructure** | Zero — no vector DB, no embedding server | Requires embedding model + vector store |
| **Query volume** | Moderate — indexing pays off at ~20+ queries | High-volume, real-time (vector lookup scales better) |
| **Domain** | Financial reports, legal contracts, technical specs | Product search, FAQ retrieval, news articles |

**Decision rule:**  
- Document is long + structured + repeatedly queried → **PageIndex**  
- Document is short or queries arrive at high velocity → **Vector RAG**  
- Both need high accuracy on complex multi-hop questions → **PageIndex**

## 13 — Summary

**End-to-end pipeline in ~10 lines:**

```python
# INDEXING (one-time)
from pageindex import build_index
index = build_index("annual_report.pdf", model="gpt-4o")   # → JSON tree

# RETRIEVAL (per query)
toc = build_toc_view(index)                                 # flatten tree to text
node_ids = llm_select_nodes(query, toc)                     # LLM picks sections
content  = fetch_content(index, node_ids)                   # pull page text
answer   = llm_generate_answer(query, content)              # LLM answers
```

**Concept map:**

| Concept | Vector RAG counterpart | Role in PageIndex |
|---|---|---|
| Tree index (JSON) | Vector DB | Retrieval data structure |
| Node summary | Chunk embedding | Retrieval signal |
| `node_id` range | Chunk position | Content extraction address |
| LLM TOC reasoning | Cosine similarity | Retrieval mechanism |
| `asyncio.gather()` summaries | Batch embedding | Parallel enrichment |
| Recursive splitting | Chunk overlap | Handles oversized sections |
| Retrieval loop (1–4 iter) | Single k-NN lookup | Multi-hop sufficiency check |

**Common failure modes:**

| Failure | Root Cause | Fix |
|---|---|---|
| Node covers too many pages | `max_pages_per_node` too large | Lower to 5–8 pages |
| Poor summaries mislead LLM navigation | Weak summary generation prompt | Add domain-specific prompt for summaries |
| LLM selects wrong node first pass | Ambiguous or terse section titles | Increase `max_iterations` to 3 |
| No TOC detected in scanned PDF | Document is image-based | Pre-process with OCR before indexing |
| Indexing cost too high | Many concurrent LLM calls | Reduce concurrency or use a cheaper model for summaries |